In [22]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# Load dataset
df = pd.read_csv('../data/all_f1_cars.csv')
print("Initial shape:", df.shape)

Initial shape: (233640, 71)


In [23]:

if len(df) > 25000:
    df = df.sample(n=25000, random_state=42).reset_index(drop=True)

# Locate position column and create binary classification target (Podium: finish <= 3)
target_col = [col for col in df.columns if 'position' in col.lower()]
target_col = target_col[0] if target_col else df.columns[-1]

df['Podium'] = (pd.to_numeric(df[target_col], errors='coerce') <= 3).astype(int)
y = df['Podium']

# Drop non-feature identifiers and keep only numeric features
drop_cols = [target_col, 'Podium'] + [col for col in df.columns if 'id' in col.lower()]
X = df.drop(columns=drop_cols, errors='ignore')
X = X.select_dtypes(include=[np.number])

# Impute missing values with column medians
X = X.fillna(X.median(numeric_only=True))

# Train / Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

results = []
print("Training shape:", X_train_scaled.shape)
print("Testing shape:", X_test_scaled.shape)
print("Target distribution:\n", y.value_counts())

Training shape: (20000, 37)
Testing shape: (5000, 37)
Target distribution:
 Podium
0    21129
1     3871
Name: count, dtype: int64


In [24]:
from sklearn.linear_model import LogisticRegression

model_lr = LogisticRegression(random_state=42, max_iter=1000)
model_lr.fit(X_train_scaled, y_train)
preds_lr = model_lr.predict(X_test_scaled)
probs_lr = model_lr.predict_proba(X_test_scaled)[:, 1]

results.append([
    "Logistic Regression",
    accuracy_score(y_test, preds_lr),
    f1_score(y_test, preds_lr, zero_division=0),
    roc_auc_score(y_test, probs_lr)
])
print("Logistic Regression completed.")

Logistic Regression completed.


In [25]:
from sklearn.svm import SVC

# Sample 2,000 rows for SVC to prevent quadratic training hang
sample_limit = min(2000, len(X_train_scaled))
X_train_svc = X_train_scaled[:sample_limit]
y_train_svc = y_train.iloc[:sample_limit]

model_svc = SVC(probability=True, random_state=42)
model_svc.fit(X_train_svc, y_train_svc)
preds_svc = model_svc.predict(X_test_scaled)
probs_svc = model_svc.predict_proba(X_test_scaled)[:, 1]

results.append([
    "SVC (Sampled)",
    accuracy_score(y_test, preds_svc),
    f1_score(y_test, preds_svc, zero_division=0),
    roc_auc_score(y_test, probs_svc)
])
print("SVC completed.")

c:\Users\unhac\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


SVC completed.


In [26]:
from sklearn.tree import DecisionTreeClassifier

model_dt = DecisionTreeClassifier(max_depth=7, random_state=42)
model_dt.fit(X_train_scaled, y_train)
preds_dt = model_dt.predict(X_test_scaled)
probs_dt = model_dt.predict_proba(X_test_scaled)[:, 1]

results.append([
    "Decision Tree",
    accuracy_score(y_test, preds_dt),
    f1_score(y_test, preds_dt, zero_division=0),
    roc_auc_score(y_test, probs_dt)
])
print("Decision Tree completed.")

Decision Tree completed.


In [27]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
model_rf.fit(X_train_scaled, y_train)
preds_rf = model_rf.predict(X_test_scaled)
probs_rf = model_rf.predict_proba(X_test_scaled)[:, 1]

results.append([
    "Random Forest",
    accuracy_score(y_test, preds_rf),
    f1_score(y_test, preds_rf, zero_division=0),
    roc_auc_score(y_test, probs_rf)
])
print("Random Forest completed.")

Random Forest completed.


In [28]:
from sklearn.ensemble import GradientBoostingClassifier

model_gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42)
model_gb.fit(X_train_scaled, y_train)
preds_gb = model_gb.predict(X_test_scaled)
probs_gb = model_gb.predict_proba(X_test_scaled)[:, 1]

results.append([
    "Gradient Boosting",
    accuracy_score(y_test, preds_gb),
    f1_score(y_test, preds_gb, zero_division=0),
    roc_auc_score(y_test, probs_gb)
])
print("Gradient Boosting completed.")

Gradient Boosting completed.


In [ ]:
from sklearn.neighbors import KNeighborsClassifier

model_knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
model_knn.fit(X_train_scaled, y_train)
preds_knn = model_knn.predict(X_test_scaled)
probs_knn = model_knn.predict_proba(X_test_scaled)[:, 1]

results.append([
    "KNN",
    accuracy_score(y_test, preds_knn),
    f1_score(y_test, preds_knn, zero_division=0),
    roc_auc_score(y_test, probs_knn)
])
print("KNN completed.")